## RQ2: 
### Considering unit traffic exposure, is the risk of fatal and serious accident associated with posted speed limit near SCATS-monitored intersections

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import pyarrow
import warnings
import geopandas as gpd
from IPython.display import clear_output
from pathlib import Path
from datetime import time
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, average_precision_score, balanced_accuracy_score)

THE_ANSWER = 42

### Output style (optional)

In [3]:
plt.style.use("tableau-colorblind10")
import matplotlib as mpl
from matplotlib import font_manager
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
if ("IBM Plex Mono" in available_fonts):
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["IBM Plex Mono"],
        "font.monospace": ["IBM Plex Mono"],
        "axes.unicode_minus": False,
        "font.size": 12,
        "figure.dpi": 120,
    })

### Data check

In [4]:
df = pd.read_parquet("rq1.parquet")
print(df.shape)

assert(df.shape == (195523, 98))

(195523, 98)


### Data overview

#### Re-fetch datetime information from `accident_rq1.csv`

In [36]:
acc_df = pd.read_csv("data/accident_rq1.csv")
acc_df["DATETIME"] = pd.to_datetime(acc_df["ACCIDENT_DATE"] + " " + acc_df["ACCIDENT_TIME"])

df = pd.read_parquet("rq1.parquet")
df = df.merge(acc_df[["ACCIDENT_NO", "DATETIME"]], on="ACCIDENT_NO", how="left")
df = df[df["DATETIME"] >= "2014-01-01"]
print(df.shape)

(167625, 99)


#### Re-fetch location information from `node.csv`

In [37]:
node_df = pd.read_csv("data/raw/node.csv")
loc_df = pd.read_csv("data/raw/accident_location.csv")
site_df = pd.read_csv("victorian_traffic_signals.csv")

loc_node = loc_df.merge(
    node_df[["NODE_ID", "LATITUDE", "LONGITUDE"]],
    on="NODE_ID",
    how="left"
)

loc_node = loc_node[
    loc_node["LATITUDE"].notna()
    & loc_node["LONGITUDE"].notna()
    & (loc_node["LATITUDE"] != 0)
    & (loc_node["LONGITUDE"] != 0)
].copy()

loc_node["_dist_sort"] = loc_node["DISTANCE_LOCATION"].fillna(float("inf"))

best_node = (
    loc_node
    .sort_values(["ACCIDENT_NO", "_dist_sort"])
    .drop_duplicates("ACCIDENT_NO", keep="first")
    .drop(columns=["_dist_sort"])
)

acc_gdf = gpd.GeoDataFrame(
    best_node[["ACCIDENT_NO", "LATITUDE", "LONGITUDE"]],
    geometry=gpd.points_from_xy(best_node["LONGITUDE"], best_node["LATITUDE"]),
    crs="EPSG:4326"
).to_crs("EPSG:7855")

site_gdf = gpd.GeoDataFrame(
    site_df[["SITE_NO", "LATITUDE", "LONGITUDE"]],
    geometry=gpd.points_from_xy(site_df["LONGITUDE"], site_df["LATITUDE"]),
    crs="EPSG:4326"
).to_crs("EPSG:7855")

# Nearest site
acc_site = gpd.sjoin_nearest(
    acc_gdf,
    site_gdf[["SITE_NO", "geometry"]],
    how="left",
    distance_col="DIST_TO_SITE_M"
)

# Only keep within 100m
acc_site_100m = acc_site[
    acc_site["DIST_TO_SITE_M"] <= 150
].copy()

# 
accident_site = (
    acc_site_100m[["ACCIDENT_NO", "SITE_NO"]]
    .drop_duplicates("ACCIDENT_NO")
    .reset_index(drop=True)
)

accident_site["SITE_NO"] = accident_site["SITE_NO"].astype("int32")

df = df.merge(
    accident_site,
    on="ACCIDENT_NO",
    how="inner"
)

print(df.shape)

df.to_parquet("rq2_acc.parquet")

(68799, 100)


#### Check DateTime range to decide how much SCATS data is needed

In [38]:
print(f"Earliest: {df["DATETIME"].min()}\nLatest: {df["DATETIME"].max()}")

Earliest: 2014-01-01 01:30:00
Latest: 2025-08-31 18:40:00


#### SCATS data starts from 2024-01-01 and can be capped at 2025-08-31

In [22]:
scat_meta_df = pd.read_csv("victorian_traffic_signals.csv")
scat_test_df = pd.read_csv("scats/VSDATA_20250831.csv")

site_ids = set(scat_meta_df["SITE_NO"].dropna().astype(int))
scats_ids = set(scat_test_df["NB_SCATS_SITE"].dropna().astype(int))

print(len(site_ids), len(scats_ids), len(site_ids & scats_ids))
coverage = len(site_ids & scats_ids) / len(scats_ids)
print(f"SCATS volume sites covered by metadata: {coverage:.2%}")

V_COLS = [f"V{i:02d}" for i in range(96)]
AM_PEAK_COLS = [f"V{i:02d}" for i in range(28, 36)]  # 07-09
PM_PEAK_COLS = [f"V{i:02d}" for i in range(64, 72)]  # 16-18

def assemble_df(csv):
    expected_cols = [
        "NB_SCATS_SITE",     # intersection number (SITE_ID)
        "QT_INTERVAL_COUNT", # date reading
        "NB_DETECTOR",       # detector number     (DETECTOR_ID)
        *V_COLS,             # volume (15 min increments)
        "NM_REGION",         # SCATS region
        "CT_RECORDS",        # count of number of 15 min where data exists
        "QT_VOLUME_24HOUR",  # sum of vol recorded within the day
        "CT_ALARM_24HOUR",   # count of alarms within the dayc 
    ]
    df = pd.read_csv(csv, engine="pyarrow")
    assert(set(expected_cols) <= set(df.columns))
    

    # Standardise ID
    df["NB_SCATS_SITE"] = pd.to_numeric(df["NB_SCATS_SITE"], errors="coerce").astype("Int64")
    df["NB_DETECTOR"] = pd.to_numeric(df["NB_DETECTOR"], errors="coerce").astype("Int64")

    # Extract date
    df["DATE"] = pd.to_datetime(
        df["QT_INTERVAL_COUNT"],
        format="%Y-%m-%d %H:%M:%S.%f",
        errors="coerce"
    ).dt.normalize()

    # Numerical vals
    numeric_cols = [
        *V_COLS,
        "CT_RECORDS",
        "QT_VOLUME_24HOUR",
        "CT_ALARM_24HOUR",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Manually calculate volume sum
    df.drop(columns=["QT_VOLUME_24HOUR"], inplace=True)
    v = df[V_COLS].mask(df[V_COLS].lt(0), np.nan)
    df["NUM_VALID_V"] = v.notna().sum(axis=1).astype("int32")
    df["SUM_VALID_V"] = v.sum(axis=1, skipna=True, min_count=1)

    # Keep raw metadata before cleaning
    keys = ["NB_SCATS_SITE", "DATE"]
    raw_meta = (
        df.groupby(keys, dropna=False)
        .agg(
            NUM_RAW_DETECTOR=("NB_DETECTOR", "nunique"),
            NUM_RAW_ROWS=("NB_DETECTOR", "size"),
        )
        .reset_index()
    )

    # Drop severely incomplete detector row
    valid = df["NUM_VALID_V"] >= 90
    df = df.loc[valid].copy()
    v = v.loc[valid]

    # Calculate per site sum
    site_v = (
        v.groupby([df["NB_SCATS_SITE"], df["DATE"]], dropna=False, observed=True)
        .sum(min_count=1)
        .reset_index()
    )

    site_v["SUM_VALID_V_SITE"] = site_v[V_COLS].sum(axis=1, skipna=True)
    site_v["V_AM_PEAK"] = site_v[AM_PEAK_COLS].sum(axis=1, skipna=True)
    site_v["V_PM_PEAK"] = site_v[PM_PEAK_COLS].sum(axis=1, skipna=True)
    site_v["V_15M_MAX"] = site_v[V_COLS].max(axis=1, skipna=True)
    site_v["V_15M_STD"] = site_v[V_COLS].std(axis=1, skipna=True)

    # Aggregate by day
    meta = (
        df.groupby(keys, dropna=False, observed=True)
        .agg(
            NUM_VALID_V=("NUM_VALID_V", "sum"),
            NUM_VALID_DETECTOR=("NB_DETECTOR", "nunique"),
            MIN_RECORDS=("CT_RECORDS", "min"),
            MEAN_RECORDS=("CT_RECORDS", "mean"),
            REGION=("NM_REGION", "first"),
        )
        .reset_index()
    )

    # Keep number of all data (including invalid)
    meta = meta.merge(
        raw_meta,
        on=keys,
        how="left"
    )

    site_day = site_v[
        ["NB_SCATS_SITE", "DATE", "SUM_VALID_V_SITE", "V_AM_PEAK", "V_PM_PEAK", "V_15M_MAX", "V_15M_STD"]
    ].merge(
        meta,
        on=keys,
        how="left"
    ).rename(columns={"NB_SCATS_SITE": "SITE_NO", "SUM_VALID_V_SITE": "SUM_VALID_V"})

    site_day = site_day.dropna(subset=["SITE_NO", "DATE"])
    site_day["SITE_NO"] = site_day["SITE_NO"].astype("int32")

    # Quality assurance
    site_day["EXPECTED_NUM_V"] = site_day["NUM_VALID_DETECTOR"] * 96

    site_day["COMPLETENESS"] = (
        site_day["NUM_VALID_V"] / site_day["EXPECTED_NUM_V"]
    )

    site_day["V_24H_ADJ"] = (
        site_day["SUM_VALID_V"]
        * site_day["EXPECTED_NUM_V"]
        / site_day["NUM_VALID_V"]
    )

    site_day.loc[
        site_day["COMPLETENESS"] < 0.95,
        "V_24H_ADJ"
    ] = np.nan

    site_day["DETECTOR_RETENTION"] = (
        site_day["NUM_VALID_DETECTOR"] / site_day["NUM_RAW_DETECTOR"]
    )

    return site_day.copy()

site_day = assemble_df("scats/VSDATA_20150101.csv")
print(site_day.shape)
print(site_day.columns)
print(site_day.describe())
print("Missing V_24H_ADJ:", site_day["V_24H_ADJ"].isna().mean())
print("Zero observed volume:", (site_day["SUM_VALID_V"] == 0).mean())
print("Low completeness:", (site_day["COMPLETENESS"] < 0.95).mean())
print("Low detector retention:", (site_day["DETECTOR_RETENTION"] < 0.8).mean())

5001 4656 4439
SCATS volume sites covered by metadata: 95.34%
(3412, 18)
Index(['SITE_NO', 'DATE', 'SUM_VALID_V', 'V_AM_PEAK', 'V_PM_PEAK', 'V_15M_MAX',
       'V_15M_STD', 'NUM_VALID_V', 'NUM_VALID_DETECTOR', 'MIN_RECORDS',
       'MEAN_RECORDS', 'REGION', 'NUM_RAW_DETECTOR', 'NUM_RAW_ROWS',
       'EXPECTED_NUM_V', 'COMPLETENESS', 'V_24H_ADJ', 'DETECTOR_RETENTION'],
      dtype='str')
           SITE_NO                 DATE    SUM_VALID_V    V_AM_PEAK  \
count  3412.000000                 3412    3412.000000  3412.000000   
mean   3075.041618  2015-01-01 00:00:00   20381.705451   717.388628   
min     100.000000  2015-01-01 00:00:00       0.000000     0.000000   
25%    1246.750000  2015-01-01 00:00:00   11084.750000   357.000000   
50%    3277.500000  2015-01-01 00:00:00   17257.500000   562.000000   
75%    4467.250000  2015-01-01 00:00:00   25977.250000   870.500000   
max    7667.000000  2015-01-01 00:00:00  111677.000000  8617.000000   
std    1763.352092                  NaN   

/tmp/ipykernel_17756/4069872662.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["NUM_VALID_V"] = v.notna().sum(axis=1).astype("int32")
/tmp/ipykernel_17756/4069872662.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["SUM_VALID_V"] = v.sum(axis=1, skipna=True, min_count=1)
/tmp/ipykernel_17756/4069872662.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1)

### Read all SCATS agg to site x day

In [23]:
site_days = []
errs = []


with warnings.catch_warnings():
    warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
    paths = sorted(Path("scats").glob("*.csv"))
    for i, csv_path in enumerate(paths):
        clear_output(wait=True)
        print(f"Loading {i}/{len(paths)} csv", flush=True)
        try:
            site_day = assemble_df(csv_path)
            site_day["SRC"] = csv_path.name
            site_days.append(site_day)
        except Exception as e:
            print(f"SRC: {csv_path.name}, ERR: {repr(e)}")
            errs.append({
                "SRC": csv_path.name,
                "ERR": repr(e)
            })

scats = pd.concat(site_days, ignore_index=True)
err_df = pd.DataFrame(errs)
print(err_df)
print(scats.shape)
err_df.to_csv("err.csv", index=False)
scats.to_parquet("scats.parquet", compression="zstd", index=False)

Loading 4251/4252 csv
Empty DataFrame
Columns: []
Index: []
(16805646, 19)


### Assign accident with SCAT IDs

In [46]:
acc_df = pd.read_parquet("rq2_acc.parquet")
scats_df = pd.read_parquet("scats.parquet")

# First aggregate accident by site-day
acc_df["FATAL_COUNT"] = (acc_df["SEVERITY"] == 1).astype("int16")
acc_df["SERIOUS_COUNT"] = (acc_df["SEVERITY"] == 2).astype("int16")
acc_df["FATAL_OR_SERIOUS_COUNT"] = (acc_df["SEVERITY"].isin([1, 2])).astype("int16")
acc_df["OTHER_INJ_COUNT"] = (acc_df["SEVERITY"] == 3).astype("int16")
acc_df["NO_INJ_COUNT"] = (acc_df["SEVERITY"] == 4).astype("int16")
acc_df["DATE"] = acc_df["DATETIME"].dt.normalize()

acc_site_day = (
    acc_df.groupby(["SITE_NO", "DATE"])
    .agg(
        ACCIDENT_COUNT = ("ACCIDENT_NO", "nunique"),
        FATAL_COUNT=("FATAL_COUNT", "sum"),
        SERIOUS_COUNT=("SERIOUS_COUNT", "sum"),
        FATAL_OR_SERIOUS_COUNT=("FATAL_OR_SERIOUS_COUNT", "sum"),
        OTHER_INJ_COUNT=("OTHER_INJ_COUNT", "sum"),
        NO_INJ_COUNT=("NO_INJ_COUNT", "sum")
    )
    .reset_index()
)

# Use mode as an indicating speed limit
def mode_or_na(x):
    m = x.dropna().mode()
    return m.iloc[0] if not m.empty else pd.NA

site_speed = (
    acc_df.groupby("SITE_NO")["SPEED_ZONE"]
    .agg(mode_or_na)
    .reset_index()
    .rename(columns={
        "SPEED_ZONE": "SITE_SPEED_ZONE_MODE"
    })
)

scats_df["DATE"] = pd.to_datetime(scats_df["DATE"]).dt.normalize()
acc_site_day["DATE"] = pd.to_datetime(acc_site_day["DATE"]).dt.normalize()

scats_df = scats_df.merge(
    acc_site_day,
    on=["SITE_NO", "DATE"],
    how="left"
)

count_cols = [
    "ACCIDENT_COUNT",
    "FATAL_COUNT",
    "SERIOUS_COUNT",
    "FATAL_OR_SERIOUS_COUNT",
    "OTHER_INJ_COUNT",
    "NO_INJ_COUNT"
]

scats_df[count_cols] = scats_df[count_cols].fillna(0)

for c in count_cols:
    scats_df[c] = scats_df[c].astype("int16")

scats_df = scats_df.merge(
    site_speed,
    on="SITE_NO",
    how="left"
)

# Remove sites without speed zone at all
scats_df = scats_df[scats_df["SITE_SPEED_ZONE_MODE"].notna()].copy()
scats_df["SITE_SPEED_ZONE_MODE"] = scats_df["SITE_SPEED_ZONE_MODE"].astype("int16")

print(scats_df)

          SITE_NO       DATE  SUM_VALID_V  V_AM_PEAK  V_PM_PEAK  V_15M_MAX  \
0             100 2014-01-01      14099.0      483.0     1818.0      300.0   
1             101 2014-01-01      12634.0      376.0     1810.0      305.0   
2             102 2014-01-01      20872.0      667.0     3399.0      557.0   
3             103 2014-01-01      10232.0      347.0     1592.0      274.0   
4             104 2014-01-01      25310.0      751.0     3914.0      812.0   
...           ...        ...          ...        ...        ...        ...   
16805629    32149 2025-08-31       5286.0      300.0      726.0      151.0   
16805630    32152 2025-08-31       8838.0      678.0     1369.0      239.0   
16805634    32160 2025-08-31      13541.0      348.0     2489.0      384.0   
16805635    32176 2025-08-31      20592.0      971.0     3212.0      569.0   
16805638    32182 2025-08-31       1424.0      106.0      186.0       49.0   

           V_15M_STD  NUM_VALID_V  NUM_VALID_DETECTOR  MIN_RECO

### Feature engineering

In [50]:
# Remove zero exposure entries since we are only studying per exposure
scats_df = scats_df[scats_df["V_24H_ADJ"] > 0].copy()

scats_df["EXPOSURE_VOLUME"] = scats_df["V_24H_ADJ"].astype("float64")
scats_df["LOG_EXPOSURE_VOLUME"] = np.log(scats_df["EXPOSURE_VOLUME"])

scats_df.to_parquet(
    "rq2.parquet",
    compression="zstd",
    index=False
)

print(scats_df.shape)
print(scats_df.dtypes)

(15923380, 28)
SITE_NO                            int32
DATE                      datetime64[ns]
SUM_VALID_V                      float64
V_AM_PEAK                        float64
V_PM_PEAK                        float64
V_15M_MAX                        float64
V_15M_STD                        float64
NUM_VALID_V                        int32
NUM_VALID_DETECTOR                 int64
MIN_RECORDS                      float64
MEAN_RECORDS                     float64
REGION                               str
NUM_RAW_DETECTOR                   int64
NUM_RAW_ROWS                       Int64
EXPECTED_NUM_V                     int64
COMPLETENESS                     float64
V_24H_ADJ                        float64
DETECTOR_RETENTION               float64
SRC                                  str
ACCIDENT_COUNT                     int16
FATAL_COUNT                        int16
SERIOUS_COUNT                      int16
FATAL_OR_SERIOUS_COUNT             int16
OTHER_INJ_COUNT                    int16
N